In [ ]:
import pandas as pd
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

## Total Returns Extended to Commodities

### Data sources
- st louis 3m tbill used as risk free rate for determination of sharpe, it can be used as approximation funding cost available since 1930
- shiller 10y bond rate and equity return and div, this allows computing 10y total return and equity total return
- spot commo prices from CMO, spot can be used approx for return on precious metals, and possibly copper. Others, notable oil spot price return is missing massive future roll return
- CL1 and CL6 historical data from bloomberg, since 1991. This allows computing future roll return, which is significant for oil.

In [ ]:
def tbillrate():
    df = pd.read_csv("data/TB3MS.csv")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df["TB3MS"] /= 100
    return df.set_index("observation_date")
def compute_gold_tr(df):
    for colname in df.columns:
        df[colname+'TR'] = df[colname]/df[colname].shift(1)
    return df
def goldprice(col):
    df = pd.read_csv("data/cmo-data-monthly.csv")
    df = df[["date"]+list(col.keys())]
    df["date"] = pd.to_datetime(df["date"])+dt.timedelta(days=1)
    df = df.loc[df["date"]>=dt.datetime(1971,2,1)]
    return compute_gold_tr(df.set_index("date").rename(columns=col)).copy()
def read_shiller_out():
    df = pd.read_csv("data/shiller_out.csv")
    df['Date'] = pd.to_datetime(df['Date'], format="%Y-%m-%d")+dt.timedelta(days=-14) # imported as 14th, will be used as 1st 
    df = df.set_index("Date")
    df = df.join(tbillrate(),how="inner")
    compute_bond_tr(df)
    compute_tbill_tr(df)
    return df.copy()
def read_shiller_cmo(col):
    return read_shiller_out().join(goldprice(col),how="inner").copy()
          
def compute_bond_tr(df):
    r = df['Rate10y']
    T = 10
    duration = -(1-np.exp(-r*T))/r
    bondcarry = r.shift(1)/12
    bondtotalret = 1+(r-r.shift(1))*duration+bondcarry
    df["bondTR"] = bondtotalret
    return bondtotalret
"""def compute_eq_tr(df):
    df['eqTR'] = df['SP500']/df['SP500'].shift(1)+df['Div']/12/df['SP500']
def compute_cpi_tr(df):
    df['cpiTR'] = 1+df['CPI'].pct_change()
    """
def getbtctr():
    with sqlite3.connect("yahoofut.db") as con:
        btcdf = pd.read_sql("select date,close from futures where ticker='BTC-USD'", con=con)
    btcdf["date"] = pd.to_datetime(btcdf["date"])
    btcdf = btcdf.set_index("date")
    btcdf.sort_index()
    btcdf = btcdf.resample('M').last()
    btcdf = btcdf.reset_index()
    btcdf["date"] = btcdf["date"]+dt.timedelta(days=1)
    btcdf = btcdf.set_index("date")
    btcdf.sort_index()
    btcdf["btcTR"] = 1+btcdf["close"].pct_change()
    btcdf = btcdf[["btcTR"]].join(tbillrate(),how="inner")
    compute_tbill_tr(btcdf)
    return btcdf

def compute_tbill_tr(df):
    df['tbTR'] = 1+df['TB3MS'].shift(1)*365/360/12 # compute with shift with last rate, as if this was a 1m rate

def max_drawdown(wealth_series):
    """Compute maximum drawdown from a wealth / total return index series"""
    w = wealth_series.dropna()
    cummax = np.maximum.accumulate(w)
    drawdown = (w - cummax) / cummax
    return drawdown.min()   # most negative value

def metrics_monthly_ret(df,cols):
    """
    Estimates constant excess return mu* using discounted log-returns
    S_t / M_t, allowing the risk-free rate r_t to vary over time.
    """
    df = df[cols].copy()
    trcolumns = [c for c in df.columns if "TR"==c[-2:] and c not in ["cpiTR","portTR"]]
    logret = np.log(df[trcolumns])          # monthly log returns
    # --- Discounted log-returns: m*_t = ln(S_t) - ln(M_t) ---
    # logret['tbTR'] = ln(M_t) - ln(M_{t-1}) = r_t * dt
    discounted_logret = logret.sub(logret['tbTR'], axis=0)  # m*_t for ALL assets
    # Drop tbTR itself (its discounted return is identically zero)
    riskycol = [c for c in trcolumns if c !='tbTR']
    discounted_logret = discounted_logret[riskycol]
    # --- Estimators on discounted series ---
    mstar_bar = discounted_logret[riskycol].mean()* 12  # annualize mean of m*_t, drift of log disc asseet 
    sigma     = discounted_logret[riskycol].std()* np.sqrt(12)   # annualized volatility
    # mu* dt = E[d ln S/M] + 0.5 * Var[ln S/M]  
    mustar = mstar_bar + 0.5 * sigma**2 
    # Average risk-free rate for reporting
    r_bar = logret['tbTR'].mean()*12                      # annualized average r_t
    # Risk premium in discrete compounding
    mustar_discrete = np.exp(mustar) - 1
    # --- Same post-processing as before ---
    data = {'mustar': mustar_discrete, 'sigma': sigma}
    data['sharpe'] = data['mustar'] / data['sigma']
    data['dd'] = max_drawdown(np.exp(np.cumsum(discounted_logret)))
    corr = discounted_logret[riskycol].corr()
    cov = discounted_logret[riskycol].cov() * 12
    try:
        wstar = np.linalg.solve(cov, data['mustar'])
    except Exception as e:
        raise
        wstar = np.nan
    K = np.sum(wstar)
    data['w'] = w = wstar / np.abs(K)
    for c in riskycol:
        data[f'rho({c[:-2]})'] = corr[c]
    metricsdf = pd.DataFrame(data, index=riskycol)
    haslong = [1 if "Carry" not in c else 0 for c in riskycol]           
    hascarry = [1 if "CarryTR" in c else 0 for c in riskycol] 
    hascarryplus = [1 if "CarryPlusTR" in c else 0 for c in riskycol]
    if len(riskycol)!=2 or np.sum(hascarry)==0 or np.sum(haslong)==0:
        if np.sum(hascarryplus)==0:
            print(riskycol)
            port_ret       = df.loc[discounted_logret.index,"tbTR"]+np.dot(np.expm1(discounted_logret), w)
            df.loc[discounted_logret.index, "portTR"]   = port_ret
            port_logret    = np.log(port_ret)
            port_mstar_bar = port_logret.mean()* 12  
            port_sigma     = port_logret.std()* np.sqrt(12) 
            port_mustar    = port_mstar_bar + 0.5 * port_sigma**2 
            port_mustar_discrete = np.exp(port_mustar) - 1
            metricsdf = pd.concat([metricsdf,pd.DataFrame({
                'mustar': [port_mustar_discrete],
                'sigma': [port_sigma],
                'sharpe': [port_mustar_discrete / port_sigma],
                'w': [np.sum(w)],
                'dd': [max_drawdown(np.exp(np.cumsum(port_logret)))]
            }, index=['portTR'])])
    return metricsdf, r_bar, K, df

def get_tr_col(columns,excl):
    return [c for c in columns if c[-2:]=="TR" and c not in excl]

def partition_df(df, K):
    n = len(df)
    split_indices = np.linspace(0, n, K+1, dtype=int)
    result = {"startdate": [], "enddate": [], "n":[], "df": []}
    for i in range(K):
        start_idx = split_indices[i]
        end_idx = split_indices[i+1]
        segment = df.iloc[start_idx:end_idx]
        result["n"].append(len(segment))
        result["startdate"].append(segment.index[0])
        result["enddate"].append(segment.index[-1])
        result["df"].append(segment)
    return result

def partition_metrics(df, K=None, excl=[]):
    K = K if K is not None else len(df)//120
    result = partition_df(df, K=K)
    result['metrics'] = []
    result['r'] = []
    result['K'] = []
    for df in result['df']:
        metricsdf,r,K,_ = metrics_monthly_ret(df,get_tr_col(df.columns,excl))
        result['metrics'].append(metricsdf[[c for c in metricsdf.columns if "rho(" not in c]])
        result['r'].append(r)
        result['K'].append(K)
    result.pop("df")
    print("mu* is excess return, dd is max dd, w is optimal (unconstrained) weight, K is kelly leverage (mu*/sigma**2). portTR = w-weighted portfolio of the assets above.")
    return result


In [ ]:

def getcolor(c):
    if "eq" in c:
        color = "blue"
    elif "bond" in c:
        color = "darkgreen"
    elif "oil" in  c:
        color = "black"
    elif "copper" in c:
        color = "darkorange"
    elif "btc" in c:
        color = "orange"
    elif "gold" in c:
        color = "gold"
    elif "silver" in c:
        color = "silver"
    elif "platinum" in c:
        color = "lightgray"
    elif "port" in c:
        color = "purple"
    return color

def show_returns(df,filename,excl):
    cols = get_tr_col(df.columns,excl)
    dfmetrics,r,K,df = metrics_monthly_ret(df,cols)
    print(dfmetrics.index)
    print(dfmetrics)
    for c in dfmetrics.index:
        ls = "--" if "Carry" in c else "-"
        ls = ":" if "CarryPlus" in c else ls
        ls = "-." if "CarryMinus" in c else ls
        wstr = f"w={dfmetrics.loc[c,'w']:.0%}" if "portTR" in dfmetrics.index else ""
        plt.plot(np.cumprod(df[c]/df["tbTR"]),color=getcolor(c), linestyle=ls,
                label=f"{c[:-2]}: $\mu^*$={dfmetrics.loc[c,'mustar']:.1%}, " \
                    f"$\sigma$={dfmetrics.loc[c,'sigma']:.0%}, " \
                    f"S={dfmetrics.loc[c,'sharpe']:.2f}, " \
                    f"dd={dfmetrics.loc[c,'dd']:.0%}, " + wstr)
    plt.legend()
    plt.title(f"Asset Total Return r={r:.1%} K={K:.1f}")
    plt.ylabel("log total return")
    plt.yscale('log')
    datesstr = f"from {str(df.index[0])[:10]} to {str(df.index[-1])[:10]}"
    plt.xlabel(datesstr)
    print(datesstr)
    print(f"r={r:.2%} K={K:.1f} (kelly leverage)")
    plt.grid(True)  
    if not filename is None:
        plt.savefig(filename)
        plt.close()
    else:
        plt.show()
    return dfmetrics

def show_all_returns(df,show,excl):
    assets = [c[:-2] for c in df.columns if "TR" in c and c not in ["cpiTR","tbTR","portTR"]]
    filename = None if show else "_".join(assets).replace(" (spot)","spot")+".png"
    dfmetrics = show_returns(df,filename,excl)
    return dfmetrics,filename



In [ ]:
r"""
## Bond total return math:
Standard Bond Math for Semi-Annual Coupon Bonds

**Price** of a bond with face value 1, annual coupon $c$, yield $y$, maturity $n$ years:
$$B(c,y) = \sum_{k=1}^{2n} \frac{c/2}{(1+y/2)^k} + \frac{1}{(1+y/2)^{2n}}$$

**Macaulay Duration** (weighted average time to cash flows, in years):
$$D_{\text{Mac}}(c,y) = \frac{1}{B} \left[ \sum_{k=1}^{2n} \frac{k}{2} \cdot \frac{c/2}{(1+y/2)^k} + n \cdot \frac{1}{(1+y/2)^{2n}} \right]$$

**Modified Duration** (price sensitivity to yield):
$$D_{\text{Mod}}(c,y) = \frac{D_{\text{Mac}}}{1+y/2}$$

**Price Change** (first-order Taylor approximation):
$$\Delta B \approx -B \cdot D_{\text{Mod}} \cdot \Delta y$$

"""

def compute_bond_duration(c, y, n=10):
    """
    Compute price and durations for semi-annual coupon bonds.
    Parameters:
    c : array of coupon rates (annual)
    y : array of yields (annual)
    n : maturity in years
    Returns:
    B : bond prices
    D_mac : Macaulay durations (years)
    D_mod : modified durations (years)
    """
    periods = 2 * n
    k = np.arange(1, periods + 1)
    # Cash flow times in years
    t = k / 2
    # Discount factors: (1 + y/2)^{-k} for each bond, each period
    discount = (1 + y[:, None]/2) ** (-k[None, :])
    # Coupon cash flows
    coupon_cf = c[:, None] / 2
    # Price
    coupon_pv = np.sum(coupon_cf * discount, axis=1)
    principal_pv = (1 + y/2) ** (-periods)
    B = coupon_pv + principal_pv
    # Macaulay duration
    weighted_coupons = np.sum(t[None, :] * coupon_cf * discount, axis=1)
    weighted_principal = n * principal_pv
    D_mac = (weighted_coupons + weighted_principal) / B
    # Modified duration
    D_mod = D_mac / (1 + y/2)
    return B, D_mac, D_mod

def compute_bond_pnl(y, c=None, n=10):
    """
    Compute bond P&L from yield changes.
    
    Parameters:
    y : array of yields (annual)
    c : array of coupon rates (optional, defaults to par bonds with c = y shifted)
    n : maturity in years
    
    Returns:
    delta_B : approximate price changes
    """
    if c is None:
        c = y.copy()  # Assume par bonds
    B, _, D_mod = compute_bond_duration(c, y, n)
    prevy = np.roll(y, 1)
    prevy[0] = np.nan
    carry = y/12
    delta_B = -B * D_mod * (y-prevy) + carry
    return 1+delta_B

# running check
df = read_shiller_out() 
delta_B = compute_bond_pnl(df["Rate10y"])
delta_Bann = compute_bond_tr(df)
bondcomp = pd.DataFrame({"bondMDwithCarryTR":delta_B,"bondAnnuityPnlTR":delta_Bann})
bondcomp["tbTR"] = df["tbTR"]
show_all_returns(bondcomp,True,excl=[])
print("we see excess return is identical within 1e03, actual note coupon which is unspecified probably introduces more difference than methodology.")

### Method
- tbill total return $\mu_r= 1+r . 365/360/12$ where $r$ is the rate in ACT.360 convention
- equity total return $\mu_e=P(i+1)/P(i) + D(i)/12$, where $P$ is price, $D$ annual dividend
- bond total return $\mu_b=A(i+1) (y(i+1)-y(i)) + y(i) / 12$ where $A$ is the annuity, $y$ the 10y yield
- commo price return $\mu_o=S(i+1)/S(i)$ where $S$ is spot price (this is total return for precious metals)
- contango amount $c=(F(i+6)-F(i+1))/F(i+6)$, contango rate $y_c=c^{1/5}$
- commo future total return $\mu_o=S(i+1)/S(i)-y_c$ 

### CMO Data
- crude oil spot shows big supply shocks in the 70s, return is significant
- gold, platinum, and silver are similar but gold has the lowest vol, 
- copper only recently started to beat inflation  from the 2000s (energiewende or China demand growing)
- softs tend to grow only 2%-3%, so return is flat above inflation, no excess return.
- algo prefers gold and copper to silver and platinum. The latter seems to be within efficient frontier of gold and copper.

### Metrics and Optimal Portfolio
- monthly log ret annualized vol $\sigma$
- monthly annualized expected return $\mu$ 
- monthly annualized expected excess return $\mu^*=\mu-r$
- Sharpe = $\mu^*/\sigma$
- optimal weightws $w^* = \Sigma^{-1} \mu^*$
- Optimal Kelly leverage $K=\sum(w)$
- normed weights $w = w^*/K$

## Shiller +Fed Data Only: Equity and Bond excess return from 1934 (92 years)

In [ ]:
dfsh = read_shiller_out()
show_all_returns(dfsh,True,[])
show_all_returns(dfsh,False,[])



In [ ]:
partition_metrics(dfsh)

### Exploring Determinants of Bond Return

In [ ]:
# def regress(x,y,xlabel,ylabel,title):
#     mask = ~(np.isnan(x) | np.isnan(y))
#     x_clean = x[mask].values
#     y_clean = y[mask].values
#     coeffs = np.polyfit(x_clean, y_clean, 1)
#     a, b = coeffs
#     y_pred = a * x_clean + b
#     errstd = np.std(y_clean - y_pred)
#     plt.scatter(x, y, alpha=0.2)
#     plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
#     plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
#              transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
#     plt.xlabel(xlabel)
#     plt.ylabel(ylabel)
#     plt.title(title)
#     plt.show()
#     return a,b
from scipy import stats
def regress(x, y, xlabel, ylabel, title):
    mask = ~(np.isnan(x) | np.isnan(y))
    x_c, y_c = x[mask].values, y[mask].values
    res = stats.linregress(x_c, y_c)
    y_pred = res.slope * x_c + res.intercept
    r2 = res.rvalue**2
    plt.scatter(x, y, alpha=0.2)
    plt.plot(x_c, y_pred, 'r-', lw=2)
    text = f'y = {res.slope:.4f}x + {res.intercept:.4f}\n' \
             f'R² = {r2:.4f}, p = {res.pvalue:.4f}\n' \
             f'SE(a) = {res.stderr:.6f}, SE(b) = {res.intercept_stderr:.6f}'
    plt.text(0.05, 0.85, text,
             transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
    plt.xlabel(xlabel); plt.ylabel(ylabel); 
    titlestr = title if "#pivot" not in title else title.replace("#pivot",f"pivot:"+f"{-res.intercept/res.slope:.2%}")
    plt.title(titlestr); 
    print("="*len(titlestr))
    print(titlestr)
    print(f"x={xlabel},y={ylabel}")
    print(text)
    plt.show()
    print(f"slope={res.slope:.6f} intercept={res.intercept:.6f} R²={r2:.4f} p={res.pvalue:.4f} SE_slope={res.stderr:.6f} SE_intercept={res.intercept_stderr:.6f}")
    return res.slope, res.intercept

In [ ]:
asset = "btc"
dfbtc = getbtctr()
netreturn = (dfbtc[asset+"TR"]-dfbtc["tbTR"])
x = netreturn.rolling(1).mean().shift(1)
y = netreturn
a,b = regress(x,y,xlabel=f"prev {asset} return",ylabel=f"{asset} return",title=f"{asset} return")
dfbtc = carrystrat_asset(dfbtc,x,y,asset,pivot=-b/a)
show_all_returns(dfbtc,show=True,excl=[])


In [ ]:

#
# 3m tbill rate vs 10y tnote
x = dfsh["TB3MS"]
y = dfsh["Rate10y"]
regress(x,y,xlabel="3M tbill rate",ylabel="10y rate vs 3m rate",title="10y rate")
#
# 3m rate vs tnote excess return 
x = (dfsh["TB3MS"]).shift(1)
y = dfsh["bondTR"]-dfsh["tbTR"]
a,b = regress(x,y,xlabel="3M tbill rate",ylabel="bond net total return",title="bond net return vs 3m rate - #pivot")
#
# 10y-3m spread vs tnote excess return 
x = (dfsh["Rate10y"]-dfsh["TB3MS"]).shift(1)
y = dfsh["bondTR"]-dfsh["tbTR"]
a,b = regress(x,y,xlabel="10y - 3m rate spread",ylabel="bond net total return",title="bond net return vs rate spread - #pivot")




In [ ]:
def regress_asset(dfsh,asset):
    x = (dfsh[asset+"TR"]-dfsh["tbTR"]).rolling(1).mean().shift(1)
    y = dfsh[asset+"TR"]-dfsh["tbTR"]
    return x,y
def carrystrat_asset(dfsh,x,y,asset,pivot):
    dfsh[asset+"CarryTR"] = dfsh["tbTR"]+y*np.where(x > pivot, 1,-1)
    dfsh[asset+"CarryPlusTR"] = dfsh["tbTR"]+y*np.where(x > pivot, 1,1e-9)
    dfsh[asset+"CarryMinusTR"] = dfsh["tbTR"]+y*np.where(x > pivot, 1e-9,-1)
    return dfsh 

assetpivot = {}

In [ ]:

asset = "bond"
x,y = regress_asset(dfsh,asset)
a,b = regress(x,y,xlabel=f"previous {asset} net total return",ylabel=f"{asset} net total return",title=f"{asset} net return momentum - #pivot")
assetpivot[asset] = -b/a
dfsh = carrystrat_asset(dfsh,x,y,asset,pivot=assetpivot[asset])
show_all_returns(dfsh,True,["eqCarryTR","eqTR"])


### Equity Strategy: momentum

In [ ]:

x = (dfsh["TB3MS"]).shift(1)
y = dfsh["eqTR"]-dfsh["tbTR"]
a,b = regress(x,y,xlabel="previous riskfree rate ",ylabel="eq net total return",title="eq net return vs riskfree rate - #pivot")

x = (dfsh["Rate10y"]-dfsh["TB3MS"]).shift(1)
y = dfsh["eqTR"]-dfsh["tbTR"]
a,b = regress(x,y,xlabel="10y-3m rate spread ",ylabel="eq net total return",title="eq net return vs 10y-3m rate spread - #pivot")


In [ ]:
def excllist(asset):
    return [asset+"CarryTR",asset+"TR",asset+"CarryMinusTR",asset+"CarryPlusTR"]

asset = "eq"
x,y = regress_asset(dfsh,"eq")
a,b = regress(x,y,xlabel=f"prev {asset} net total return",ylabel=f"{asset} net total return",title=f"{asset} net return momentum - #pivot")
assetpivot[asset] = -b/a
dfsh = carrystrat_asset(dfsh,x,y,asset,pivot=assetpivot[asset])
show_all_returns(dfsh,True,excllist("bond"))


In [ ]:
def removecarryplus(cols):
    return [c for c in cols if "CarryPlus" in c or "CarryMinus" in c]

show_all_returns(dfsh,True,["eqTR","bondTR"]+removecarryplus(dfsh.columns))


In [ ]:
partition_metrics(dfsh,excl=["eqTR","bondTR"]+removecarryplus(dfsh.columns))

## Shiller + Fed + CMO Data: Gold since 1971 (65 years of data)

In [ ]:
col = {"CRUDE_DUBAI":"oil (spot)", "COPPER":"copper (spot)"}
for c in ["GOLD"]:
    col[c] = c.lower()
dfcmo = read_shiller_cmo(col)
dfmetrics,filename = show_all_returns(dfcmo,show=True,excl=[])
print(dfmetrics.to_markdown(floatfmt=".2%"))


In [ ]:
col = {"CRUDE_DUBAI":"oil (spot)", "COPPER":"copper (spot)"}
for c in ["GOLD","PLATINUM","SILVER"]:
    col[c] = c.lower()
dfcmo = read_shiller_cmo(col)
dfmetrics,filename = show_all_returns(dfcmo,show=True,excl=[])
print(dfmetrics.to_markdown(floatfmt=".2%"))


In [ ]:
asset = "gold"
col = {}
for c in ["GOLD","PLATINUM","SILVER"]:
    col[c] = c.lower()
dfcmo = read_shiller_cmo(col)
x,y = regress_asset(dfcmo,asset)
a,b = regress(x,y,xlabel=f"prev {asset} net total return",ylabel=f"{asset} net total return",title=f"{asset} net return momentum - #pivot")
assetpivot[asset] =-b/a
dfcmo = carrystrat_asset(dfcmo,x,y,asset,pivot=assetpivot[asset])
show_all_returns(dfcmo,True,[c for c in dfcmo.columns if asset not in c and c!="tbTR"])


In [ ]:
x,y = regress_asset(dfcmo,"eq")
dfcmo = carrystrat_asset(dfcmo,x,y,"eq",assetpivot["eq"])
x,y = regress_asset(dfcmo,"bond")
dfcmo = carrystrat_asset(dfcmo,x,y,"bond",assetpivot["bond"])

allcarry = [c for c in dfcmo.columns if "Carry" in c]
dfmetrics,filename = show_all_returns(dfcmo,show=True,excl=allcarry)
print(dfmetrics.to_markdown(floatfmt=".2%"))
partition_metrics(dfsh,excl=allcarry)

In [ ]:
asset = "silver"
# dfcmo = read_shiller_cmo({asset.upper():asset})
x,y = regress_asset(dfcmo,asset)
a,b = regress(x,y,xlabel=f"prev {asset} net total return",ylabel=f"{asset} net total return",title=f"{asset} net return momentum - #pivot")
assetpivot[asset] =-b/a
dfcmo = carrystrat_asset(dfcmo,x,y,asset,pivot=assetpivot[asset])
show_all_returns(dfcmo,True,[c for c in dfcmo.columns if asset not in c and c!="tbTR"])


In [ ]:
asset = "platinum"
x,y = regress_asset(dfcmo,asset)
a,b = regress(x,y,xlabel=f"prev {asset} net total return",ylabel=f"{asset} net total return",title=f"{asset} net return momentum - #pivot")
assetpivot[asset] =-b/a
dfcmo = carrystrat_asset(dfcmo,x,y,asset,pivot=assetpivot[asset])
show_all_returns(dfcmo,True,[c for c in dfcmo.columns if asset not in c and c!="tbTR"])


## Shiller + Fed + CMO + Bloomberg Data: from 1991 (35y)

In [ ]:
def getbbdata(tick):
    dfcl = pd.read_csv(f"data/{tick.lower()}.csv")
    dfcl["date"] = pd.to_datetime(dfcl["date"])
    dfcl = dfcl.set_index("date")
    dfcl = dfcl.resample('M').last()
    dfcl = dfcl.reset_index()
    dfcl["date"] = dfcl["date"]+dt.timedelta(days=1)
    dfcl = dfcl.set_index("date")
    # Primary axis: CL1 and CL6
    fig, ax1 = plt.subplots(figsize=(10, 6))
    color = "black" if tick=="CL" else "darkorange"
    ax1.plot(dfcl[tick+"1"], label=tick+"1", color=color)
    ax1.plot(dfcl[tick+"6"], label=tick+"6", color=color, linestyle="--")
    ax1.set_ylabel("price")
    ax1.tick_params(axis='y', labelcolor=color)
    # Secondary axis: CL6 - CL1 (in gray)
    ax2 = ax1.twinx()
    ax2.plot(dfcl[tick+"6"]/dfcl[tick+"1"], label=f"{tick}6/{tick}1", color="gray", linestyle="--", alpha=0.8)
    ax2.set_ylabel("contango", color="gray")
    ax2.axhline(y=0,color="gray")
    ax2.tick_params(axis='y', labelcolor="gray")
    # Legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    plt.title(f"{tick} Price and Contango")
    plt.grid(True, alpha=0.3)
    plt.show()
    return dfcl

# monthly contango c:
# (c6/c1)^(1/5)01 = (1+c)
def addcontango(dfcl,tick):
    contango = ((dfcl[tick+"6"]/dfcl[tick+"1"])**(1/5)-1)-dfcl["TB3MS"]*365/360/12 # net carry
    totret   = dfcl[tick+"6"].pct_change()-contango.shift(1) # excess return mu*_o=mu_o-mu_
    #dfcl[f"{tickmap[tick]}CarryTR"] = dfcl["tbTR"]+(totret-1)*np.where(contango<pivot,1,-1) 
    dfcl[f"{tickmap[tick]}TR"] = 1+totret
    dfcl[tick+"contango"] = contango
    return dfcl

tickmap = {"CL":"oil","HG":"copper"}
tick="CL"
dfcl = getbbdata(tick).join(getbbdata(tick="HG"))
precious = ["gold","silver","platinum"]
# col = {}
# for c in precious:
#     col[c.upper()] = c
# df = read_shiller_cmo(col)
# dfcl = dfcl.join(df)
dfcl = dfcl.join(dfcmo)



In [ ]:
tick = "CL"
asset = tickmap[tick]
dfcl = addcontango(dfcl,tick)
x = -dfcl[f"{tick}contango"]
y = dfcl[f"{asset}TR"]-1
a,b = regress(x,y,xlabel=f"{asset} backwardation",ylabel=f"{asset} net total return",title=f"{asset} total return with future yield - #pivot")
assetpivot[asset] =-b/a
carrystrat_asset(dfcl,x,y,asset,pivot=assetpivot[asset])
show_all_returns(dfcl,True,[c for c in dfcl.columns if c!="tbTR" and asset not in c])


In [ ]:
tick = "HG"
asset = tickmap[tick]
dfcl = addcontango(dfcl,tick)
x = -dfcl[f"{tick}contango"]
y = dfcl[f"{asset}TR"]-1
a,b = regress(x,y,xlabel=f"{asset} backwardation",ylabel=f"{asset} net total return",title=f"{asset} total return with future yield - #pivot")
assetpivot[asset] =-b/a
carrystrat_asset(dfcl,x,y,asset,pivot=assetpivot[asset])
show_all_returns(dfcl,True,[c for c in dfcl.columns if c!="tbTR" and asset not in c])


## Long only strat with future roll for oil and copper

In [ ]:
dfmetrics,filename = show_all_returns(dfcl,show=True,excl=[c for c in dfcl.columns if "Carry" in c])


In [ ]:
dfbtc = getbtctr()[["btcTR"]]
dfmetrics,filename = show_all_returns(dfcl.join(dfbtc,how="inner"),show=True,excl=[c for c in dfcl.columns if "Carry" in c])


In [ ]:
dfmetrics,filename = show_all_returns(dfcl,show=True,excl=[c for c in dfcl.columns if c[-7:]!="CarryTR" and c!="tbTR"])
# print(dfmetrics.to_markdown(floatfmt=".2%"))

In [ ]:
partition_metrics(dfcl,excl=[c for c in dfcl.columns if c[-7:]!="CarryTR" and c!="tbTR"])

In [ ]:
excl = [c for c in dfcl.columns if c[-7:]!="CarryTR" and c!="tbTR" or "silver" in c or "platinum" in c]
dfmetrics,filename = show_all_returns(dfcl,show=True,excl=excl)
dirname = '../projects/futcarry/'
dfmetrics.to_csv(dirname+"metrics-carry.csv")


In [ ]:
partition_metrics(dfcl,excl=excl)

In [ ]:
import json
with open(dirname+'pivot.json', 'w') as f:
    json.dump(assetpivot, f, indent=2)
assetpivot

In [ ]:
dfbtc = getbtctr()[["btcTR"]]
dfmetrics,filename = show_all_returns(dfcl.join(dfbtc,how="inner"),show=True,excl=excl)